# CBCT ToothSeg teacher multi-case student training

Objective: use multiple ToothSeg/FDI-style teacher labelmaps to train a browser-sized
YOLOv8 segmentation student that separates adjacent teeth better than post-processing alone.

This notebook:

1. Mounts Drive and locates paired CBCT image + ToothSeg label volumes.
2. Exports each teacher-labeled volume into YOLO instance segmentation slices.
3. Trains a single-class tooth instance model for browser inference.
4. Exports ONNX and evaluates held-out cases with `--label-mode nonzero` and `yolo-seeds`.

Runtime: T4 GPU or better.

## 1. Setup

In [ ]:
!nvidia-smi -L || true

In [ ]:
!pip -q install ultralytics SimpleITK scikit-image scipy opencv-python-headless onnx onnxslim onnxruntime pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

In [ ]:
import csv, json, os, re, shutil, subprocess, time
from pathlib import Path
import pandas as pd
from ultralytics import YOLO

PROJECT = Path('/content/drive/MyDrive/Projects/Health/CBCT')
WORK = PROJECT / 'cbct-notebook'
OUT = PROJECT / 'cbct-outputs'

EXPORTER = WORK / 'export_toothseg_yolo_slices.py'
COMPARE = WORK / 'compare_tooth_yolo_onnx_colab.py'
EXP_NAME = 'toothseg-teacher-multicase-1cls'
DATA = Path('/content/data') / EXP_NAME
RUN_PROJECT = Path('/content/runs')
RUN_NAME = EXP_NAME
DST = OUT / EXP_NAME

TARGET_SPACING = 0.3
CT_WINDOW = (-113.8, 4021)
TRAIN_AXES = ['z', 'y', 'x']
STRIDE = 3
MIN_AREA = 25
SIMPLIFY_STEP = 3
VAL_EVERY = 5
EPOCHS = 60
IMG_SIZE = 512
BATCH = 16
USE_HELDOUT = True

# Prefer the current browser-training checkpoint. The one-scan teacher checkpoint is
# a fallback, not the first choice, because it did not beat baseline by itself.
BASE_PT_CANDIDATES = [
    OUT / 'fdi-1cls' / 'best.pt',
    OUT / 'toothseg-teacher-1cls' / 'best.pt',
    Path('yolov8n-seg.pt'),
]
BASE_ONNX_CANDIDATES = [
    OUT / 'fdi-1cls' / 'best.onnx',
    WORK / 'baseline-tooth-yolov8n-seg.onnx',
    OUT / 'toothseg-teacher-1cls' / 'best.onnx',
]

for required in [PROJECT, WORK, OUT, EXPORTER, COMPARE]:
    assert required.exists(), f'Missing required path: {required}'
DST.mkdir(parents=True, exist_ok=True)
print('PROJECT:', PROJECT)
print('WORK:', WORK)
print('OUT:', OUT)
print('DST:', DST)

## 3. Find CBCT image + ToothSeg label pairs

The manual pair below is the Aug 2025 case we have already used. Add more dictionaries
to `MANUAL_PAIRS` if discovery does not find all of your teacher outputs.

In [ ]:
MANUAL_PAIRS = [
    {
        'case': 'cbct_aug2025',
        'image': str(WORK / 'teacher-toothseg' / 'cbct_aug2025_0000.nii.gz'),
        'labels': str(WORK / 'teacher-toothseg' / 'cbct_aug2025_toothseg_recovered.nii.gz'),
    },
]

# Put new precomputed teacher outputs in any of these folders, or add them above.
SEARCH_ROOTS = [
    WORK / 'teacher-toothseg',
    WORK / 'teacher-toothseg-cases',
    PROJECT / 'teacher-toothseg',
    PROJECT / 'cases',
    OUT / 'teacher-toothseg',
]

LABEL_HINTS = ('label', 'labels', 'toothseg', 'recovered', 'prediction', 'pred', 'seg')

def strip_nii_suffix(path: Path) -> str:
    name = path.name
    if name.endswith('.nii.gz'):
        return name[:-7]
    if name.endswith('.nii'):
        return name[:-4]
    return path.stem

def normalized_case_key(path: Path) -> str:
    key = strip_nii_suffix(path).lower()
    for token in [
        '_0000', '-0000', '_image', '-image', '_ct', '-ct',
        '_labels', '-labels', '_label', '-label',
        '_toothseg', '-toothseg', '_recovered', '-recovered',
        '_prediction', '-prediction', '_pred', '-pred',
        '_segmentation', '-segmentation', '_seg', '-seg',
    ]:
        key = key.replace(token, '')
    key = re.sub(r'[^a-z0-9]+', '_', key).strip('_')
    return key

def is_label_file(path: Path) -> bool:
    name = path.name.lower()
    return any(hint in name for hint in LABEL_HINTS) and '_0000' not in name

def discover_pairs():
    images_by_key = {}
    labels_by_key = {}
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for path in root.rglob('*.nii*'):
            if not path.is_file():
                continue
            key = normalized_case_key(path)
            if is_label_file(path):
                labels_by_key.setdefault(key, []).append(path)
            else:
                images_by_key.setdefault(key, []).append(path)
    found = []
    for key in sorted(set(images_by_key) & set(labels_by_key)):
        image = sorted(images_by_key[key], key=lambda p: ('_0000' not in p.name, len(str(p))))[0]
        labels = sorted(labels_by_key[key], key=lambda p: ('recovered' not in p.name.lower(), len(str(p))))[0]
        found.append({'case': key, 'image': str(image), 'labels': str(labels), 'source': 'discovered'})
    return found

pairs_by_case = {}
for item in discover_pairs():
    pairs_by_case[item['case']] = item
for item in MANUAL_PAIRS:
    case = item['case']
    pairs_by_case[case] = {**item, 'source': 'manual'}

pairs = []
for item in sorted(pairs_by_case.values(), key=lambda x: x['case']):
    image = Path(item['image'])
    labels = Path(item['labels'])
    if image.exists() and labels.exists():
        pairs.append(item)
    else:
        print('Skipping missing pair:', item)

assert pairs, 'No CBCT/ToothSeg pairs found. Add paths to MANUAL_PAIRS.'
print(f'Found {len(pairs)} usable teacher pair(s).')
display(pd.DataFrame(pairs))

## 4. Split train and evaluation cases

In [ ]:
if USE_HELDOUT and len(pairs) >= 2:
    holdout_n = max(1, round(len(pairs) * 0.2))
else:
    holdout_n = 0

if holdout_n:
    train_pairs = pairs[:-holdout_n]
    eval_pairs = pairs[-holdout_n:]
else:
    train_pairs = pairs
    eval_pairs = pairs

print('train cases:', [p['case'] for p in train_pairs])
print('eval cases:', [p['case'] for p in eval_pairs])
if len(pairs) < 5:
    print('Note: fewer than 5 cases. This can prove the pipeline, but it is still weak evidence for promotion.')

## 5. Export teacher labels into a YOLO instance dataset

In [ ]:
shutil.rmtree(DATA, ignore_errors=True)
case_summaries = []
for pair in train_pairs:
    cmd = [
        'python', str(EXPORTER),
        '--volume', pair['image'],
        '--labels', pair['labels'],
        '--output-dir', str(DATA),
        '--axes', *TRAIN_AXES,
        '--stride', str(STRIDE),
        '--min-area', str(MIN_AREA),
        '--simplify-step', str(SIMPLIFY_STEP),
        '--val-every', str(VAL_EVERY),
        '--single-class',
        '--label-mode', 'nonzero',
        '--target-spacing', str(TARGET_SPACING),
        '--ct-window', str(CT_WINDOW[0]), str(CT_WINDOW[1]),
        '--case-id', pair['case'],
    ]
    print('EXPORT', pair['case'])
    subprocess.run(cmd, check=True)
    summary = json.loads((DATA / 'summary.json').read_text())
    summary['case'] = pair['case']
    case_summaries.append(summary)

manifest = {
    'generatedAt': time.strftime('%Y-%m-%d %H:%M:%S'),
    'trainPairs': train_pairs,
    'evalPairs': eval_pairs,
    'caseSummaries': case_summaries,
    'totalSlices': sum(s['sliceCount'] for s in case_summaries),
    'totalObjects': sum(s['objectCount'] for s in case_summaries),
}
(DATA / 'multi-case-summary.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps({k: manifest[k] for k in ['totalSlices', 'totalObjects']}, indent=2))
print('data.yaml:', DATA / 'data.yaml')

## 6. Train the browser-sized student

In [ ]:
def first_existing(paths):
    for path in paths:
        p = Path(path)
        if str(p) == 'yolov8n-seg.pt' or p.exists():
            return str(p)
    raise FileNotFoundError(paths)

base_pt = first_existing(BASE_PT_CANDIDATES)
print('base training weights:', base_pt)
model = YOLO(base_pt)
model.train(
    data=str(DATA / 'data.yaml'),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    workers=2,
    patience=15,
    lr0=0.001,
    close_mosaic=10,
    project=str(RUN_PROJECT),
    name=RUN_NAME,
    exist_ok=True,
)

## 7. Export ONNX and save the candidate

In [ ]:
weights_dir = RUN_PROJECT / RUN_NAME / 'weights'
best_pt = weights_dir / 'best.pt'
assert best_pt.exists(), best_pt
model = YOLO(str(best_pt))
model.export(format='onnx', imgsz=IMG_SIZE, opset=12, simplify=True)

DST.mkdir(parents=True, exist_ok=True)
for name in ['best.pt', 'best.onnx', 'last.pt']:
    src = weights_dir / name
    if src.exists():
        shutil.copy2(src, DST / name)
for name in ['args.yaml', 'results.csv']:
    src = RUN_PROJECT / RUN_NAME / name
    if src.exists():
        shutil.copy2(src, DST / name)
shutil.copy2(DATA / 'multi-case-summary.json', DST / 'dataset-summary.json')
print('saved candidate to', DST)

## 8. Evaluate held-out cases against ToothSeg teacher labels

The comparison uses `--label-mode nonzero` and `--separation-mode yolo-seeds`, matching
the current browser-side path. Promotion should depend on fewer merges/duplicates, not Dice alone.

In [ ]:
reports_dir = DST / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

model_args = []
for baseline in BASE_ONNX_CANDIDATES:
    if Path(baseline).exists():
        model_args += ['--model', f'baseline={baseline}']
        break
student_onnx = DST / 'best.onnx'
assert student_onnx.exists(), student_onnx
model_args += ['--model', f'student={student_onnx}']

rows = []
for pair in eval_pairs:
    report = reports_dir / f"{pair['case']}-comparison-nonzero.json"
    cmd = [
        'python', str(COMPARE),
        '--image', pair['image'],
        '--labels', pair['labels'],
        '--output', str(report),
        '--conf', '0.15',
        '--iou', '0.45',
        '--mask-threshold', '0.7',
        '--core-threshold', '7',
        '--min-voxels', '8000',
        '--label-mode', 'nonzero',
        '--separation-mode', 'yolo-seeds',
    ] + model_args
    print('EVAL', pair['case'])
    subprocess.run(cmd, check=True)
    data = json.loads(report.read_text())
    for item in data['models']:
        m = item['metrics']
        rows.append({
            'case': pair['case'],
            'model': item['name'],
            'dice': m['voxelDice'],
            'precision': m['voxelPrecision'],
            'recall': m['voxelRecall'],
            'predInstances': m['predInstanceCount'],
            'gtTeeth': m['gtToothCount'],
            'matchedGtTeeth': m['matchedGtTeeth'],
            'falsePositiveInstances': m['falsePositiveInstances'],
            'duplicateInstances': m.get('duplicateInstances'),
            'mergedComponents': len(m.get('mergedComponents', [])),
            'report': str(report),
        })

df = pd.DataFrame(rows)
df.to_csv(DST / 'comparison-summary.csv', index=False)
(DST / 'comparison-summary.json').write_text(json.dumps(rows, indent=2))
display(df)
display(df.groupby('model')[['dice', 'precision', 'recall', 'predInstances', 'falsePositiveInstances', 'mergedComponents']].mean(numeric_only=True))
print('summary:', DST / 'comparison-summary.csv')

## 9. Promotion rule

Keep the candidate only if it improves the current browser baseline on held-out cases:

- fewer merged components
- no increase in duplicate or false-positive instances
- Dice and recall stable or better
- predicted instance count moves toward the teacher tooth count

If those are true, copy `best.onnx` from `DST` back into the app as
`public/models/tooth-yolov8n-seg.onnx` and rerun the local browser comparison.